# PySpark from basics to fast: fix the query, not the cluster

Part 1 walks through the PySpark fundamentals: the session, reading data, columns, missing values, filters, aggregations, SQL, pivots, lazy evaluation, caching and a first model. Part 2 runs the article's experiments on 5 million synthetic clickstream events, each comparing a slow query shape with a fast one.

Companion notebook to the article *Spark Is "100x Faster." So Why Is Your PySpark Job Still Slow?*

**Before you start:** run `python lab/gen_data.py` once from the repo root to create the dataset (about 30 seconds).

In [1]:
import os, sys, time
os.chdir("lab")            # data/ lives here
sys.path.insert(0, ".")
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
UA = "Mozilla/5.0 (Linux; Android 14; Pixel 8) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Mobile Safari/537.36"
import pandas as pd
from common import session, plan_text

spark = session("notebook")
events = lambda: spark.read.parquet("data/events")
products = lambda: spark.read.parquet("data/products")

def timed(label, fn, runs=3):
    fn()                                   # warm-up
    t = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    t.sort(); print(f"{label:<28} {'median' if runs > 1 else '1 run '} {t[len(t)//2]:6.2f}s")

def show_nodes(df, names):
    p = plan_text(df)
    print("plan nodes:", [n for n in names if n in p] or "none of", names if not [n for n in names if n in p] else "")

print("Spark", spark.version, "| events:", events().count(), "| products:", products().count())

Spark 4.0.1 | events: 5000000 | products: 2000


# Part 1. The fundamentals, in the order you will use them

Why Spark exists, in one paragraph: a single machine reads data from one disk with a few CPU cores. Hadoop MapReduce split work across many machines but wrote every intermediate result back to disk. Spark keeps intermediate data in memory where it can, and plans the whole job before running it. That is where the famous "up to 100x faster" line came from: iterative work such as machine learning, which rereads the same data many times.

How a Spark application is organised:
- **Driver**: your Python program plus the `SparkSession`. It builds the plan and hands out tasks.
- **Executors**: worker processes that run the tasks and hold cached data.
- **Cluster manager**: YARN, Kubernetes or standalone. It gives the driver its executors.

Here, `local[2]` runs the driver and executors as threads in one process, with 2 cores. The code is identical on a real cluster.

## 1.1 The SparkSession is your entry point
`SparkSession.builder...getOrCreate()` returns the running session if one exists, otherwise starts one. The helper in `lab/common.py` does exactly that.

In [2]:
spark

## 1.2 Reading data: let Spark guess, or tell it the schema
Reading a CSV without a schema makes every column a string. `inferSchema` guesses types but needs an extra pass over the data. On real pipelines, declare the schema.

In [3]:
# Write a small CSV sample so we can practise CSV options
(events().limit(50_000).coalesce(1).write.mode("overwrite")
   .option("header", True).csv("data/events_sample_csv"))

raw = spark.read.option("header", True).csv("data/events_sample_csv")
raw.printSchema()          # everything is a string

schema = ("product_id STRING, event_id STRING, session_id STRING, event_type STRING, "
          "event_ts LONG, price DOUBLE")
typed = spark.read.option("header", True).schema(schema).csv("data/events_sample_csv")
typed.printSchema()        # real types, no extra pass

root
 |-- product_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- price: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_ts: long (nullable = true)
 |-- price: double (nullable = true)



## 1.3 Looking at a DataFrame
`show`, `count`, `select` and `describe`. Note that `count` and `show` are **actions**: each one runs a job.

In [4]:
print("rows:", typed.count())
typed.select("event_type", "price").show(5)
typed.describe("price").show()
print(typed.dtypes)

rows: 50000
+-----------+------+
| event_type| price|
+-----------+------+
|   purchase| 55.24|
|   purchase|140.86|
|   wishlist| 140.7|
|add_to_cart|167.62|
|add_to_cart| 176.7|
+-----------+------+
only showing top 5 rows


+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|             50000|
|   mean| 99.78986940000024|
| stddev|57.768521023627954|
|    min|               0.0|
|    max|            199.99|
+-------+------------------+

[('product_id', 'string'), ('event_id', 'string'), ('session_id', 'string'), ('event_type', 'string'), ('event_ts', 'bigint'), ('price', 'double')]


## 1.4 Changing columns
Every transformation returns a **new** DataFrame. Nothing is changed in place.

In [5]:
df = (typed
      .withColumn("event_time", F.from_unixtime("event_ts").cast("timestamp"))
      .withColumn("price_int", F.col("price").cast("int"))
      .withColumnRenamed("event_type", "action")
      .drop("session_id"))
df.show(3, truncate=False)

+----------+---------+--------+----------+------+-------------------+---------+
|product_id|event_id |action  |event_ts  |price |event_time         |price_int|
+----------+---------+--------+----------+------+-------------------+---------+
|p-2       |e-2500000|purchase|1789022091|55.24 |2026-09-10 06:34:51|55       |
|p-3       |e-2500001|purchase|1789260228|140.86|2026-09-13 00:43:48|140      |
|p-3       |e-2500002|wishlist|1788757888|140.7 |2026-09-07 05:11:28|140      |
+----------+---------+--------+----------+------+-------------------+---------+
only showing top 3 rows


## 1.5 Missing values
Drop rows with nulls, or fill them. Shown on a tiny DataFrame so the effect is visible.

In [6]:
messy = spark.createDataFrame(
    [("p-1", 10.0, "view"), ("p-2", None, "purchase"), (None, 5.0, None)],
    "product_id STRING, price DOUBLE, event_type STRING")
messy.na.drop().show()                                   # any null drops the row
messy.na.drop(subset=["price"]).show()                   # only look at price
messy.na.fill({"price": 0.0, "event_type": "unknown"}).show()

+----------+-----+----------+
|product_id|price|event_type|
+----------+-----+----------+
|       p-1| 10.0|      view|
+----------+-----+----------+



+----------+-----+----------+
|product_id|price|event_type|
+----------+-----+----------+
|       p-1| 10.0|      view|
|      NULL|  5.0|      NULL|
+----------+-----+----------+



+----------+-----+----------+
|product_id|price|event_type|
+----------+-----+----------+
|       p-1| 10.0|      view|
|       p-2|  0.0|  purchase|
|      NULL|  5.0|   unknown|
+----------+-----+----------+



## 1.6 Filtering
`filter` and `where` are the same. Combine conditions with `&` (and), `|` (or) and `~` (not), each wrapped in brackets.

In [7]:
df.filter((F.col("action") == "purchase") & (F.col("price") > 150)).show(3)
df.where(~F.col("action").isin("view", "search")).groupBy("action").count().show()
print(df.filter("price < 5 AND action = 'checkout'").count())   # a SQL string works too

+----------+---------+--------+----------+------+-------------------+---------+
|product_id| event_id|  action|  event_ts| price|         event_time|price_int|
+----------+---------+--------+----------+------+-------------------+---------+
|       p-3|e-2500007|purchase|1790349375|176.15|2026-09-25 15:16:15|      176|
|     p-616|e-2500083|purchase|1790466277|196.26|2026-09-26 23:44:37|      196|
|       p-2|e-2500086|purchase|1789677342|162.28|2026-09-17 20:35:42|      162|
+----------+---------+--------+----------+------+-------------------+---------+
only showing top 3 rows


+-----------+-----+
|     action|count|
+-----------+-----+
|   wishlist| 8360|
|   purchase| 8181|
|add_to_cart| 8352|
|   checkout| 8425|
+-----------+-----+



195


## 1.7 Grouping, aggregating and sorting

In [8]:
(df.groupBy("action")
   .agg(F.count("*").alias("events"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.countDistinct("product_id").alias("products"))
   .orderBy(F.desc("events"))
   .show())

+-----------+------+---------+--------+
|     action|events|avg_price|products|
+-----------+------+---------+--------+
|   checkout|  8425|    99.53|    1149|
|     search|  8377|   100.27|    1130|
|   wishlist|  8360|    99.47|    1147|
|add_to_cart|  8352|   100.83|    1124|
|       view|  8305|     99.4|    1134|
|   purchase|  8181|    99.21|    1102|
+-----------+------+---------+--------+



## 1.8 SQL on a DataFrame
Register a temporary view and query it. Same engine, same plan, different syntax.

In [9]:
df.createOrReplaceTempView("events_sample")
spark.sql("""
    SELECT action, ROUND(SUM(price), 2) AS revenue
    FROM events_sample
    GROUP BY action
    ORDER BY revenue DESC
""").show()

+-----------+---------+
|     action|  revenue|
+-----------+---------+
|add_to_cart|842153.07|
|     search|839990.37|
|   checkout|838581.79|
|   wishlist|831589.15|
|       view|825514.02|
|   purchase|811665.07|
+-----------+---------+



## 1.9 Pivot
Pass the pivot values explicitly. Otherwise Spark runs an extra job just to find them.

In [10]:
(df.withColumn("band", F.when(F.col("price") < 50, "low").otherwise("high"))
   .groupBy("action").pivot("band", ["low", "high"]).count()
   .show())

+-----------+----+----+
|     action| low|high|
+-----------+----+----+
|   wishlist|2158|6202|
|   purchase|2103|6078|
|add_to_cart|2042|6310|
|       view|2120|6185|
|     search|2056|6321|
|   checkout|2159|6266|
+-----------+----+----+



## 1.10 Transformations, actions and lazy evaluation
- **Transformations** (`select`, `filter`, `withColumn`, `groupBy`, `join`) only add steps to a plan.
- **Actions** (`count`, `show`, `collect`, `write`) run the plan.

Spark builds the full plan first, optimises it, then runs it. Part 2 starts by measuring this.

## 1.11 Caching, collect and toPandas: use with care
Each action recomputes the plan from the source. `cache()` keeps the result in memory for reuse, and it only fills on the next action. `collect()` and `toPandas()` pull **everything** to the driver, so use them only on small results.

In [11]:
hot = events().filter(F.col("event_type") == "purchase").cache()
t0 = time.perf_counter(); hot.count(); print(f"first count (reads and caches): {time.perf_counter()-t0:.2f}s")
t0 = time.perf_counter(); hot.count(); print(f"second count (from cache):      {time.perf_counter()-t0:.2f}s")
print(hot.groupBy("event_type").count().toPandas())   # fine: the result is one row
hot.unpersist()

first count (reads and caches): 3.03s
second count (from cache):      0.09s


  event_type   count
0   purchase  833511


DataFrame[product_id: string, event_id: string, session_id: string, event_type: string, event_ts: bigint, price: double]

## 1.12 A machine learning model in a few lines
Spark ML wants all features in **one vector column**, and `VectorAssembler` builds it. The data is synthetic and random, so the model learns nothing useful. The point is the mechanics.

In [12]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

ml = (typed.withColumn("label", (F.col("event_type") == "purchase").cast("double"))
           .withColumn("hour", F.hour(F.from_unixtime("event_ts")).cast("double")))
features = VectorAssembler(inputCols=["price", "hour"], outputCol="features").transform(ml)
train, test = features.randomSplit([0.8, 0.2], seed=42)
model = LogisticRegression(featuresCol="features", labelCol="label").fit(train)
print("coefficients:", model.coefficients)
model.transform(test).select("price", "hour", "label", "prediction").show(3)

coefficients: [-0.0003671949876901285,-0.00045092158739749627]


+------+----+-----+----------+
| price|hour|label|prediction|
+------+----+-----+----------+
| 86.44|11.0|  0.0|       0.0|
|198.18|10.0|  0.0|       0.0|
|101.49| 2.0|  0.0|       0.0|
+------+----+-----+----------+
only showing top 3 rows


# Part 2. Why your PySpark job is still slow

The article's experiments, in the same order. Each pairs a slow query shape with a fast one, times both, and shows the plan node that gives the slow shape away.

## 0. Lazy evaluation: the slow line is not the slow line
Transformations only build a plan. The action at the end pays for everything. `explain()` shows the route before you drive it.

In [13]:
t0 = time.perf_counter()
q = (events().filter(F.col("event_type") == "purchase")
       .withColumn("price_band", (F.col("price") / 50).cast("int"))
       .groupBy("product_id", "price_band").agg(F.sum("price").alias("revenue")))
print(f"building the plan: {time.perf_counter() - t0:.2f}s")
t0 = time.perf_counter(); q.count(); print(f"running it:        {time.perf_counter() - t0:.2f}s")
q.explain(mode="formatted")   # look for PushedFilters on the scan

building the plan: 0.14s


running it:        1.19s
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [product_id#1105, event_type#1108, price#1110]
Batched: true
Location: InMemoryFileIndex [file:lab/data/events]
PushedFilters: [IsNotNull(event_type), EqualTo(event_type,purchase)]
ReadSchema: struct<product_id:string,event_type:string,price:double>

(2) Filter
Input [3]: [product_id#1105, event_type#1108, price#1110]
Condition : (isnotnull(event_type#1108) AND (event_type#1108 = purchase))

(3) Project
Output [3]: [product_id#1105, price#1110, cast((price#1110 / 50.0) as int) AS price_band#1112]
Input [3]: [product_id#1105, event_type#1108, price#1110]

(4) HashAggregate
Input [3]: [product_id#1105, price#1110, price_band#1112]
Keys [2]: [product_id#1105, price_band#1112]
Functions [1]: [partial_sum(price#1110)]
Aggregate Attributes [1]:

## 1. Let the small table travel: broadcast hash join vs sort merge join

In [14]:
q = lambda: events().join(products(), "product_id").groupBy("category").agg(F.count("*").alias("n"))
show_nodes(q(), ["BroadcastHashJoin", "SortMergeJoin"]); timed("broadcast (default)", lambda: q().collect())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
show_nodes(q(), ["BroadcastHashJoin", "SortMergeJoin"]); timed("forced sort merge", lambda: q().collect())
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

plan nodes: ['BroadcastHashJoin'] 


broadcast (default)          median   1.40s


plan nodes: ['SortMergeJoin'] 


forced sort merge            median   2.74s


## 2. The OR that turned a join into a loop
An OR across two columns has no single equality key, so Spark falls back to a nested loop. Rewrite as one equi-join per key.

In [15]:
def ev2():
    return events().limit(1_000_000).withColumn("alt_sku", F.regexp_replace("product_id", "^p-", "sku-"))

def slow():
    e, lk = ev2(), products()
    return e.join(lk, (e.product_id == lk.product_id) | (e.alt_sku == lk.alt_sku), "left").agg(F.countDistinct(lk.brand))

def fast():
    e, lk = ev2(), products()
    a = e.join(lk.select("product_id", F.col("brand").alias("b1")), "product_id", "left")
    return a.join(lk.select("alt_sku", F.col("brand").alias("b2")), "alt_sku", "left") \
            .agg(F.countDistinct(F.coalesce("b1", "b2")))

names = ["BroadcastNestedLoopJoin", "BroadcastHashJoin"]
show_nodes(slow(), names); timed("OR condition", lambda: slow().collect())
show_nodes(fast(), names); timed("two equi-joins", lambda: fast().collect())
print("same answer:", slow().first()[0] == fast().first()[0])

plan nodes: ['BroadcastNestedLoopJoin'] 


OR condition                 median  19.13s
plan nodes: ['BroadcastHashJoin'] 


two equi-joins               median   1.70s


same answer: True


## 3. Python UDFs leave the engine

In [16]:
# Pull the numeric part out of a product id such as "p-1234"
@F.pandas_udf(IntegerType())
def num_pandas(s: pd.Series) -> pd.Series:
    return s.str.split("-").str[1].astype("int32")

@F.udf(IntegerType())
def num_python(s):
    return None if s is None else int(s.split("-")[1])

native = lambda c: F.regexp_extract(c, r"(\d+)$", 1).cast("int")
for label, fn in [("native", native), ("pandas_udf", num_pandas), ("python_udf", num_python)]:
    q = lambda fn=fn: events().select("product_id").withColumn("bucket", fn(F.col("product_id")) % 10).groupBy("bucket").count()
    show_nodes(q(), ["BatchEvalPython", "ArrowEvalPython"]); timed(label, lambda: q().collect())

plan nodes: none of ['BatchEvalPython', 'ArrowEvalPython']


native                       median   1.34s
plan nodes: ['ArrowEvalPython'] 


pandas_udf                   median   4.50s
plan nodes: ['BatchEvalPython'] 


python_udf                   median   5.24s


## 4. Deduplicate on the key, not the whole row

In [17]:
def wide():
    ev = events()
    return (ev.unionByName(ev.sample(0.1, seed=3))
              .withColumn("user_agent", F.concat(F.lit(UA + " sid="), F.col("session_id")))
              .withColumn("utm", F.struct(F.lit("newsletter").alias("source"), F.lit("email").alias("medium"),
                                          F.concat(F.lit("autumn-sale-"), F.col("event_type")).alias("campaign"))))
timed("dropDuplicates() all columns", lambda: wide().dropDuplicates().count(), runs=1)  # 1 run: slow
timed("dropDuplicates(['event_id'])", lambda: wide().dropDuplicates(["event_id"]).count(), runs=1)
# Compare "Shuffle Write" for the two jobs in the Spark UI (Stages tab).

dropDuplicates() all columns 1 run   12.34s


dropDuplicates(['event_id']) 1 run    5.01s


## 5. Filter the partition, not a disguise of it

In [18]:
day = "2026-08-30"
by_dt = lambda: spark.read.parquet("data/events_by_dt")
hidden = lambda: by_dt().filter(F.to_date(F.from_unixtime("event_ts")) == F.lit(day)).agg(F.sum("price"))
direct = lambda: by_dt().filter(F.col("dt") == F.lit(day)).agg(F.sum("price"))
for label, q in [("expression on timestamp", hidden), ("partition column", direct)]:
    print(label, "| partition filter used:", "PartitionFilters: [isnotnull(dt" in plan_text(q()))
    timed(label, lambda q=q: q().collect())

expression on timestamp | partition filter used: False


expression on timestamp      median   1.35s


partition column | partition filter used: True


partition column             median   0.15s


## 6. The one configuration win: Adaptive Query Execution (on by default)

In [19]:
q = lambda: events().groupBy("product_id", "event_type").agg(F.countDistinct("session_id").alias("n"))
spark.conf.set("spark.sql.adaptive.enabled", "false"); timed("AQE off, 200 partitions", lambda: q().collect(), runs=1)
spark.conf.set("spark.sql.adaptive.enabled", "true");  timed("AQE on (default)", lambda: q().collect(), runs=1)

AQE off, 200 partitions      1 run   17.40s


AQE on (default)             1 run    6.33s


## 7. Skew, and the 256 MB threshold nobody reads
A partition only counts as skewed above 5x the median **and** above 256 MB. On this data, the automatic split never fires until you lower the threshold.

In [20]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
q = lambda: events().join(products(), "product_id").groupBy("category", "event_type").agg(F.sum("price"))
d = q(); d.collect(); print("defaults        -> skew split applied:", "skew=true" in plan_text(d))
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "4MB")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "4MB")
d = q(); d.collect(); print("threshold 4 MB  -> skew split applied:", "skew=true" in plan_text(d))
for k in ["spark.sql.autoBroadcastJoinThreshold", "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    spark.conf.unset(k)

defaults        -> skew split applied: False


threshold 4 MB  -> skew split applied: True


## Read the plan, change the question
Before you change configuration, run `explain(mode="formatted")` and look for:

| Plan node | What it means | Fix |
|---|---|---|
| `BroadcastNestedLoopJoin`, `CartesianProduct` | Join with no usable equality key | Rewrite as equi-joins |
| `SortMergeJoin` against a small table | Size estimate is wrong | Check statistics, measure, then hint `broadcast()` |
| `BatchEvalPython`, `ArrowEvalPython` | Python in the hot path | Native expression first |
| More `Exchange` nodes than expected | Extra shuffles and stages | Ask whether each one is needed |
| No `PartitionFilters` / `PushedFilters` | Reading data you throw away | Filter the raw partition column |

In [21]:
spark.stop()